# 05 · Planner (LLM)

**Role of this layer.** The planner turns a request into a plan, which is inert
JSON. It is one of only two places an LLM is used (the other is the narrator).

* The LLM sees the **generated** capability summary, dataset **metadata** and
  rules, never data values. The request is wrapped in `<user_request>` tags and
  treated as data, not as instructions.
* Output arrives through a `submit_plan` tool whose input schema is the plan
  schema; it is then parsed strictly (G01) and validated like any other plan.
* Modes: **live** (call the API), **cached** (reuse a cached plan for an
  identical request, prompt, model and registry), **mock** (canned plans for the
  five demo requests; no API key needed).

This notebook uses live mode if the configured provider's API key is set
(`QUANTLAB_LLM_PROVIDER`, default `openai`, with `OPENAI_API_KEY`), otherwise mock.

In [1]:
import pandas as pd

from quantlab.config import load_settings

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)
settings = load_settings()

In [2]:
from quantlab.data import load_catalog
from quantlab.plan import describe_plan, plan_tool_schema
from quantlab.planner import PROMPT_VERSION, build_system_prompt, cache_key, generate_plan
from quantlab.scenarios import SCENARIOS_BY_ID
from quantlab.validation import fill_defaults, validate_plan

catalog = load_catalog(settings.catalog_path)
mode = "live" if settings.has_api_key else "mock"
print(
    "mode:",
    mode,
    "| provider:",
    settings.llm_provider,
    "| model:",
    settings.planner_model,
    "| prompt:",
    PROMPT_VERSION,
)

mode: mock | provider: openai | model: gpt-6-luna | prompt: planner-v2


The start of the system prompt (the registry summary and dataset metadata follow):

In [3]:
prompt = build_system_prompt(catalog, ["NASDAQCOM"])
print(prompt[:1800])
print(f"... ({len(prompt):,} characters in total)")

You are the planner of QuantLab, a quantitative analysis assistant used by a bank's markets and risk teams. You turn a user's request into a PLAN: an ordered list of steps, each calling one capability from the approved registry below.

You never write or run code. You only choose registered capabilities and fill in their parameters. A deterministic validator checks your plan, a human reviews it, and a deterministic engine runs it.

Rules:
1. Use only capabilities listed in CAPABILITY REGISTRY, with exactly the input names, parameter names, types, ranges and choices shown. Omit an optional parameter to use its default.
2. Number steps s1, s2, s3, ... Each input refers to the id of an EARLIER step, for example {"series": "s1"}.
3. Load data with load_series, using only dataset ids from DATASETS.
4. Respect what series mean: compute returns of prices and indices, but difference rates and yields; resample levels before computing returns or changes; give spread two inputs with the same freq

In [4]:
schema = plan_tool_schema()
print("tool schema fields:", list(schema["properties"]))
print("step fields:", list(schema["properties"]["steps"]["items"]["properties"]))

tool schema fields: ['dataset_ids', 'steps', 'cannot_fulfill', 'refusal_reason', 'planner_notes']
step fields: ['id', 'capability', 'inputs', 'params', 'rationale']


Planning scenario 1:

In [5]:
scenario = SCENARIOS_BY_ID["volatility_flag_success"]
result = generate_plan(scenario.request, list(scenario.dataset_ids), mode, settings, catalog)
print(
    "provider:",
    result.provider,
    "| served by:",
    result.served_model,
    "| from cache:",
    result.from_cache,
    "| latency ms:",
    result.latency_ms,
)
print(result.error or describe_plan(result.plan))
if result.plan:
    print("validation:", validate_plan(fill_defaults(result.plan), catalog).codes())

provider: mock | served by: mock | from cache: False | latency ms: 0.0
s1 load_series(dataset_id=NASDAQCOM)
s2 resample(series=s1, freq=W, how=last)
s3 compute_returns(series=s2, method=simple, periods=1)
s4 rolling_volatility(series=s3, window=12, annualize=True)
s5 flag_regime(series=s4, method=expanding, quantile=0.9, min_periods=52)
validation: ['G12']


A request for a model (scenario 3) never runs: there is no capability for model
fitting or prediction, so the planner refuses (G13) and says what is possible.

In [6]:
prediction = SCENARIOS_BY_ID["random_forest_block"]
result_c = generate_plan(prediction.request, list(prediction.dataset_ids), mode, settings, catalog)
plan_c = result_c.plan
if plan_c is None:
    print("no plan:", result_c.error)
elif plan_c.cannot_fulfill:
    print("refused:", plan_c.refusal_reason)
else:
    print(describe_plan(plan_c))
    print("validation:", validate_plan(fill_defaults(plan_c), catalog).codes())

refused: There is no registered capability for model fitting or prediction (such as a random forest). The registry can instead: load a dataset, resample it to weekly or monthly, compute returns or differences, take a spread, compute rolling volatility, flag high-volatility periods, summarise a series, and compare a series between flagged and normal periods.


A prompt injection (scenario 4) ends in a refusal, never in code being run:

In [7]:
injection = SCENARIOS_BY_ID["prompt_injection_block"]
result_d = generate_plan(injection.request, list(injection.dataset_ids), mode, settings, catalog)
plan_d = result_d.plan
print("cannot_fulfill:", plan_d.cannot_fulfill if plan_d else None)
print("reason:", plan_d.refusal_reason if plan_d else result_d.error)

cannot_fulfill: True
reason: The request was treated as data, not as instructions: it asks to ignore the rules and run Python code, and it contains no analysis the registry supports. QuantLab only runs registered capabilities and never executes code.


The plan cache key covers everything that could change the answer: request,
datasets, model, prompt version and registry version.

In [8]:
print(cache_key(scenario.request, list(scenario.dataset_ids), settings.planner_model))
print(cache_key(scenario.request, list(scenario.dataset_ids), "another-model"))

a99ddc0ee63e7e0ca51e3f6742adb218dc58357e87053f3bdf7fd377f7696314
faae0039953b80fbcda873776ddae4f452d037cf57087b69d1e0dcff8e380076
